In [ ]:
import sys
from pathlib import Path
import pandas as pd
import torch
from torch.utils.data import DataLoader

REPO_ROOT = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "common" / "paths.py").is_file()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from common.paths import ROOT

from common.dataset import PairedElasticityDataset
from common.paths import DATASET_CONFIG, RAW_ZIP, cache_dir
from common.preprocess import prepare_dataset
from common.split import assert_patient_exclusivity

DATASET_NAME = "parotid"
SEED = 42
CACHE_DIR = cache_dir(DATASET_NAME)
frame, external_frame = prepare_dataset(
    RAW_ZIP,
    DATASET_CONFIG[DATASET_NAME],
    CACHE_DIR,
    seed=SEED,
)
assert_patient_exclusivity(frame)
display(frame.head())
display(frame.groupby("split")["patient_id"].nunique().rename("patients"))
if external_frame is not None:
    display(external_frame.head())

In [ ]:
train_frame = frame.loc[frame["split"] == "train"].copy()
validation_frame = frame.loc[frame["split"] == "val"].copy()
test_frame = frame.loc[frame["split"] == "test"].copy()
train_dataset = PairedElasticityDataset(train_frame, augment=True)
validation_dataset = PairedElasticityDataset(validation_frame, augment=False)
test_dataset = PairedElasticityDataset(test_frame, augment=False)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=0)
validation_loader = DataLoader(validation_dataset, batch_size=4, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=0)
batch = next(iter(train_loader))
batch_structure = {
    key: {"type": type(value).__name__, "shape": tuple(value.shape), "dtype": str(value.dtype)}
    if torch.is_tensor(value)
    else {"type": type(value).__name__, "length": len(value), "sample": list(value[:2])}
    for key, value in batch.items()
}
display(batch_structure)
display({"gray_min": float(batch["gray"].min()), "gray_max": float(batch["gray"].max()), "emean": batch["emean"][:4].tolist()})